<a href="https://colab.research.google.com/github/ttienthuy131/Greenwashing-Research/blob/main/greenwashing_text_extraction_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Sustainability Report Environmental Disclosure Extractor**
**bold text**

This notebook:

1.   Extracts environmental disclosure statements from corporate
sustainability reports (PDF format).
2.   Reads PDF reports using a two-column parsing method to preserve the correct reading order.
3. Cleans and normalizes extracted text to remove formatting issues such as line breaks, extra spaces, and soft hyphens.

4. Splits report text into individual sentences using punctuation-based sentence segmentation.

5. Applies keyword-based matching to identify sentences related to predefined environmental indicators.

6. Classifies detected sentences into thematic ESG groups (e.g., Environmental Efficiency, Supply Chain, Green Products, Governance).

7. Labels each sentence as green practice or green communication based on indicator classification.

8. Records metadata including indicator code, keyword, page number, and extracted sentence.

9. Outputs a structured dataset for environmental disclosure analysis and potential greenwashing measurement.

In [ ]:
!pip install pdfplumber


In [ ]:
# =======================
# FULL CODE: Extract 1 sentence containing each keyword (by indicator)
# - Reads PDF in 2 columns (left then right) to avoid cross-column mixing
# - Splits into sentences and keeps only the sentence that contains keyword
# - Outputs Excel + CSV
# =======================

import re
from pathlib import Path
import pdfplumber
import pandas as pd

# =======================
# 1) INPUT / OUTPUT
# =======================
PDF_PATH = r"/content/HM-Group-Annual-and-sustainability-report-2024.pdf"  # đổi nếu cần
OUT_XLSX = "hm_extracted_keywords.xlsx"
OUT_CSV  = "hm_extracted_keywords.csv"

# =======================
# 2) INDICATORS (UPDATED KEYWORDS)
# =======================
INDICATORS = [
    # 🔹 Environmental Efficiency (EE)
    {"code": "EE_01", "group": "EE",
     "indicator": "Material consumption, efficiency and recycling (0–6)",
     "keywords": ["material"], "category": "green practice"},

    {"code": "EE_02", "group": "EE",
     "indicator": "Energy consumption and efficiency (0–6)",
     "keywords": ["energy"], "category": "green practice"},

    {"code": "EE_03", "group": "EE",
     "indicator": "Water consumption and efficiency (0–6)",
     "keywords": ["water"], "category": "green practice"},

    {"code": "EE_04", "group": "EE",
     "indicator": "Effluent and waste control, treatment and recycling (0–6)",
     "keywords": ["waste"], "category": "green practice"},

    {"code": "EE_05", "group": "EE",
     "indicator": "Biodiversity preservation and ecological restoration (0–6)",
     "keywords": ["biodiversity"], "category": "green practice"},

    {"code": "EE_06", "group": "EE",
     "indicator": "Exhaust gases control and treatment (0–6)",
     "keywords": ["emissions"], "category": "green practice"},

    # 🔹 Investment & Spending (IS)
    {"code": "IS_01", "group": "IS",
     "indicator": "Investment in innovations and R&D related to environmental sustainability (0–1)",
     "keywords": ["environmental innovation"], "category": "green practice"},

    {"code": "IS_02", "group": "IS",
     "indicator": "Expenditure on waste disposal and remediation (0–1)",
     "keywords": ["waste disposal"], "category": "green practice"},

    {"code": "IS_03", "group": "IS",
     "indicator": "Expenditure on prevention and environmental management (0–1)",
     "keywords": ["environmental management"], "category": "green practice"},

    # 🔹 Environmental Supply Chain (ES)
    {"code": "ES_01", "group": "ES",
     "indicator": "Environmental impact assessment on suppliers (0–1)",
     "keywords": ["supplier environmental assessment"], "category": "green practice"},

    {"code": "ES_02", "group": "ES",
     "indicator": "Adopt environmental standards to select new suppliers (0–1)",
     "keywords": ["green procurement"], "category": "green practice"},

    {"code": "ES_03", "group": "ES",
     "indicator": "Terminating the partnership with suppliers with significant negative environmental impact (0–1)",
     "keywords": ["supplier termination"], "category": "green practice"},

    {"code": "ES_04", "group": "ES",
     "indicator": "Improvement of suppliers' environmental impact after environmental impact assessment (0–1)",
     "keywords": ["supplier environmental improvement"], "category": "green practice"},

    # 🔹 Green Products & Services (GP)
    {"code": "GP_01", "group": "GP",
     "indicator": "Description of the extent to which the environmental impact of products and services was reduced (0–1)",
     "keywords": ["environmental impact reduction"], "category": "green practice"},

    {"code": "GP_02", "group": "GP",
     "indicator": "Adopting green technology and product line to reduce negative environmental impact (0–1)",
     "keywords": ["green technology"], "category": "green practice"},

    {"code": "GP_03", "group": "GP",
     "indicator": "Developing, producing and selling reusable and eco-friendly products (0–1)",
     "keywords": ["eco-friendly products"], "category": "green practice"},

    {"code": "GP_04", "group": "GP",
     "indicator": "Recovery of sold products and packaging materials (0–1)",
     "keywords": ["product take-back"], "category": "green practice"},

    # 🔹 Governance (GV)
    {"code": "GV_01", "group": "GV",
     "indicator": "Environmental objectives and targets (0–1)",
     "keywords": ["environmental targets"], "category": "green communication"},

    {"code": "GV_02", "group": "GV",
     "indicator": "Environmental strategy, values, and codes of conduct (0–1)",
     "keywords": ["environmental strategy"], "category": "green communication"},

    {"code": "GV_03", "group": "GV",
     "indicator": "Formal organisation structure for environmental management (0–1)",
     "keywords": ["environmental management structure"], "category": "green communication"},

    # 🔹 Environmental Performance & Disclosure (EP)
    {"code": "EP_01", "group": "EP",
     "indicator": "Firm's compliance with environmental laws, regulations and standards (0–1)",
     "keywords": ["environmental compliance"], "category": "green communication"},

    {"code": "EP_02", "group": "EP",
     "indicator": "External audit is implemented on environmental information disclosure (0–1)",
     "keywords": ["external environmental audit"], "category": "green communication"},

    {"code": "EP_03", "group": "EP",
     "indicator": "Environmental information is disclosed in accordance with global reporting standards (0–1)",
     "keywords": ["ESG disclosure"], "category": "green communication"},

    {"code": "EP_04", "group": "EP",
     "indicator": "The company has won green company awards (0–1)",
     "keywords": ["green awards"], "category": "green communication"},

    {"code": "EP_05", "group": "EP",
     "indicator": "Overview of environmental impact of business operations, products and services (0–1)",
     "keywords": ["environmental impact"], "category": "green communication"},

    # 🔹 Environmental Implementation (EI)
    {"code": "EI_01", "group": "EI",
     "indicator": "Adopt green human resource management or environmental training (0–1)",
     "keywords": ["environmental training"], "category": "green communication"},

    {"code": "EI_02", "group": "EI",
     "indicator": "Existence of environmental accident response programmes (0–1)",
     "keywords": ["environmental emergency response"], "category": "green communication"},

    {"code": "EI_03", "group": "EI",
     "indicator": "Existence of internal environmental incentive and punishment programmes (0–1)",
     "keywords": ["environmental incentives"], "category": "green communication"},

    {"code": "EI_04", "group": "EI",
     "indicator": "Internal certification of environmental programmes (0–1)",
     "keywords": ["ISO 14001"], "category": "green communication"},

    {"code": "EI_05", "group": "EI",
     "indicator": "Periodic reviews and self-assessment of corporate environmental performance (0–1)",
     "keywords": ["environmental performance review"], "category": "green communication"},

    # 🔹 Stakeholder Involvement (SI)
    {"code": "SI_01", "group": "SI",
     "indicator": "Engagement with government and public agencies (0–1)",
     "keywords": ["government engagement"], "category": "green communication"},

    {"code": "SI_02", "group": "SI",
     "indicator": "Engagement with NGOs (0–1)",
     "keywords": ["NGO engagement"], "category": "green communication"},

    {"code": "SI_03", "group": "SI",
     "indicator": "Participation in global agreements (0–1)",
     "keywords": ["global agreements"], "category": "green communication"},

    {"code": "SI_04", "group": "SI",
     "indicator": "Engagement with suppliers and other partners (0–1)",
     "keywords": ["stakeholder engagement"], "category": "green communication"},

    {"code": "SI_05", "group": "SI",
     "indicator": "Engagement with investors (0–1)",
     "keywords": ["investor engagement"], "category": "green communication"},
]

# =======================
# 3) TEXT HELPERS
# =======================
def normalize_text(t: str) -> str:
    t = (t or "").replace("\u00ad", "")  # soft hyphen
    t = t.replace("\r", "\n")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n+", "\n", t)
    return t.strip()

def split_sentences(text: str):
    """
    Split into sentences, keep punctuation-based splits.
    Protect decimals like 29.5.
    """
    text = normalize_text(text)
    text = re.sub(r"(\d)\.(\d)", r"\1<DEC>\2", text)  # protect decimal
    # newline đơn -> space
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)

    parts = re.split(r"(?<=[\.\!\?\;])\s+|\n+", text)
    out = []
    for p in parts:
        p = p.strip().replace("<DEC>", ".")
        if len(p) >= 20:
            out.append(p)
    return out

# =======================
# 4) KEYWORD PATTERNS
# =======================
def build_patterns(indicators):
    """
    Return:
      - patterns_by_code: {code: compiled_regex_for_that_indicator}
      - meta_by_code: {code: dict(group, indicator, keyword)}
    """
    patterns_by_code = {}
    meta_by_code = {}

    for item in indicators:
        code = item["code"]
        kw = item["keywords"][0]  # mỗi chỉ tiêu 1 keyword rút gọn
        # strict word boundary; allow whitespace between words
        pat = r"(?i)\b" + re.escape(kw).replace(r"\ ", r"\s+") + r"\b"
        patterns_by_code[code] = re.compile(pat)

        meta_by_code[code] = {
            "Code": code,
            "Group": item["group"],
            "Indicator": item["indicator"],
            "Keyword": kw,
            "Category": item["category"]
        }

    return patterns_by_code, meta_by_code

PATTERNS_BY_CODE, META_BY_CODE = build_patterns(INDICATORS)

# =======================
# 5) READ PAGE IN 2 COLUMNS (LEFT THEN RIGHT)
# =======================
def extract_two_columns_text(page) -> str:
    """
    Crop page into left/right halves to avoid mixing columns.
    This works even if the page is actually single-column.
    """
    w, h = float(page.width), float(page.height)
    mid = w / 2.0

    left = page.crop((0, 0, mid, h))
    right = page.crop((mid, 0, w, h))

    t_left = left.extract_text(use_text_flow=True) or ""
    t_right = right.extract_text(use_text_flow=True) or ""

    # separate columns with blank line to avoid sentence gluing
    return "\n\n".join([t for t in [t_left, t_right] if t.strip()])

# =======================
# 6) MAIN EXTRACT
# =======================
pdf_path = Path(PDF_PATH)
if not pdf_path.exists():
    raise FileNotFoundError(f"Không thấy file: {pdf_path.resolve()}")

rows = []

with pdfplumber.open(str(pdf_path)) as pdf:
    for page_no, page in enumerate(pdf.pages, start=1):
        page_text = extract_two_columns_text(page)
        if not page_text.strip():
            continue

        for sent in split_sentences(page_text):
            # check each indicator's single keyword
            for code, pat in PATTERNS_BY_CODE.items():
                if pat.search(sent):
                    meta = META_BY_CODE[code]
                    rows.append({
                        "Code": meta["Code"],
                        "Group": meta["Group"],
                        "Indicator": meta["Indicator"],
                        "Keyword": meta["Keyword"],
                        "Category": meta["Category"],
                        "Page": page_no,
                        "Sentence": sent
                    })

df = pd.DataFrame(rows).drop_duplicates()

df.to_excel(OUT_XLSX, index=False)
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print(f"DONE: {len(df)} rows")
print(f"Saved Excel: {Path(OUT_XLSX).resolve()}")
print(f"Saved CSV  : {Path(OUT_CSV).resolve()}")


DONE: 507 rows
Saved Excel: /content/hm_extracted_keywords.xlsx
Saved CSV  : /content/hm_extracted_keywords.csv
